# XGBoost Classifier: Predict the Percent Chance of Rain Happening

In [107]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost
from imblearn.combine import SMOTEENN
from scipy.constants import precision
import optuna
import optuna.visualization as vis
import plotly  

from sklearn.metrics import r2_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler


from joblib import dump, load

In [108]:
data_frame = pd.read_csv('weather_data_excel.csv')


In [109]:
data_frame['Date'] = pd.to_datetime(data_frame['Date'])

In [110]:
data_frame['Month'] = data_frame['Date'].dt.month
data_frame['Day'] = data_frame['Date'].dt.day

In [111]:
# This is the threshold, in mm, for if rainfall is classified as rain or not
# 0.254mm is the threshold the National Weather Service uses when determine if precipitation should be reported as rain
threshold = 0.254
data_frame['Rainfall'] = (data_frame['Precipitation_mm'] >= threshold).astype(int)

In [112]:
data_frame_pos = data_frame[data_frame['Rainfall']==1]
data_frame_neg = data_frame[data_frame['Rainfall']==0]

num = data_frame.shape[0]
num_pos = data_frame_pos.shape[0]
num_neg = data_frame_neg.shape[0]

print('Number of examples = ', num)
print('Number of positive examples = ', num_pos)
print('Number of negative examples = ', num_neg)
print(f'Percentage of positive examples = {np.round((num_pos / num) * 100, 4)}%')

Number of examples =  1000000
Number of positive examples =  975735
Number of negative examples =  24265
Percentage of positive examples = 97.5735%


In [113]:
data_frame.sample(5)

,Location,Date_Time,Temperature_C,Humidity_pct,Precipitation_mm,Wind_Speed_kmh,Temperature_F,Date,Military_Time,Rain_Category,Rain_Cat_Num,Precipitation_in,Lat,Long,Month,Day,Rainfall
302800,Philadelphia,4/4/2024 1:45,14.674227,69.053557,3.117412,12.503579,58.413608,2024-04-04,01:45,Moderate Rain,1,0.122733,39.9526,75.1652,4,4,1
587086,Phoenix,4/13/2024 0:54,5.690946,59.936744,5.981417,27.108093,42.243703,2024-04-13,00:54,Moderate Rain,1,0.235489,33.4484,112.0740,4,13,1
662684,San Diego,1/9/2024 15:35,10.674694,30.931704,6.740942,20.485032,51.214449,2024-01-09,15:35,Moderate Rain,1,0.265391,32.7157,117.1611,1,9,1
694523,San Diego,2/14/2024 20:17,-1.612218,73.295140,7.061364,2.361418,29.098008,2024-02-14,20:17,Moderate Rain,1,0.278006,32.7157,117.1611,2,14,1
988776,Phoenix,3/11/2024 17:37,7.740572,78.819720,9.996889,16.017974,45.933030,2024-03-11,17:37,Heavy Rain,2,0.393578,33.4484,112.0740,3,11,1


In [114]:
y=data_frame['Rainfall']

In [115]:
X = data_frame.drop(columns=['Precipitation_in', 'Precipitation_mm', 'Date', 'Military_Time', 'Rain_Category', 'Rain_Cat_Num', 'Location', 'Rainfall', 'Date_Time', 'Temperature_C', 'Location'])


In [116]:
X.columns


Index(['Humidity_pct', 'Wind_Speed_kmh', 'Temperature_F', 'Lat', 'Long',
       'Month', 'Day'],
      dtype='object')

In [117]:
X.sample(5)


,Humidity_pct,Wind_Speed_kmh,Temperature_F,Lat,Long,Month,Day
186739,54.634551,20.670871,99.150426,32.7767,96.7970,4,25
639678,46.868478,25.250259,32.425423,29.7601,95.3701,1,5
834759,41.595965,8.805635,99.887366,29.7601,95.3701,5,9
315247,55.806553,23.805502,76.484979,37.3387,121.8853,3,8
11793,75.038771,11.550053,82.391121,40.7128,74.0060,4,1


In [118]:
y.sample(5)


821451    1
124483    1
488307    1
148901    1
399874    1
Name: Rainfall, dtype: int64

In [119]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 4700)

In [120]:
scaler = StandardScaler()
X_scaler = scaler.fit_transform(X_train)

In [121]:
# Combine both Over and Under Sampling
smote_enn = SMOTEENN(random_state=4700)
X_resampled, y_resampled = smote_enn.fit_resample(X_scaler, y_train)

## Create UnTuned XGBoost Model

In [122]:
xgb_c = xgboost.XGBClassifier( random_state = 4700)


In [123]:
xgb_c.fit(X_resampled, y_resampled)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=4700, ...)

In [124]:
y_train_pred = xgb_c.predict(X_resampled)

train_precision_score =   precision_score(y_resampled, y_train_pred)
train_recall_score = recall_score(y_resampled, y_train_pred)
train_f1_score = f1_score(y_resampled, y_train_pred)

print('Training Scores')
print('Precision Score:', train_precision_score)
print('Recall Score:', train_recall_score)
print('F1 Score:', train_f1_score)

Training Scores
Precision Score: 0.8195848619449817
Recall Score: 0.9995428633320644
F1 Score: 0.9006626508617905


In [125]:
y_test_pred = xgb_c.predict(X_test)

test_precision_score =   precision_score(y_test, y_test_pred)
test_recall_score = recall_score(y_test, y_test_pred)
test_f1_score = f1_score(y_test, y_test_pred)

print('Testing Scores')
print('Precision Score:', test_precision_score)
print('Recall Score:', test_recall_score)
print('F1 Score:', test_f1_score)

Testing Scores
Precision Score: 0.9759340106002481
Recall Score: 0.963193892973333
F1 Score: 0.9695221003450036


In [126]:
y_resampled.sample(20)

398534     0
1299682    1
483295     0
426884     0
981021     1
577165     0
311969     0
501293     0
1164793    1
917210     1
340817     0
1325695    1
682879     0
936404     1
1339134    1
1384815    1
1229946    1
594111     0
437403     0
645858     0
Name: Rainfall, dtype: int64

## Implement Optuna Hyper Parameter Tuning

In [127]:
def objective(trial):
    # Define hyperparameters to optimize
    objective = 'binary:logistic'
    max_depth = trial.suggest_int('max_depth', 1, 60)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 1)
    reg_lambda = trial.suggest_float('reg_lambda', 0.00, 10.0)

    params = {
        'objective': objective,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'eval_metric': 'logloss',
        'tree_method': 'approx',
        "reg_lambda": reg_lambda,
    }

    
    X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2)
    
    
    dtrain = xgboost.DMatrix(X_train, label=y_train)
    dval = xgboost.DMatrix(X_val, label=y_val)
    
    
    evals = [(dtrain, 'train'), (dval, 'validation')]

    
    model = xgboost.train(params, 
                          dtrain, 
                          num_boost_round=100, 
                          evals=evals, 
                          early_stopping_rounds=10, 
                          verbose_eval=False)

    
    predictions_val = model.predict(dval)
    predictions_binary_val = [1 if p > 0.5 else 0 for p in predictions_val]

    
    f1 = f1_score(y_val, predictions_binary_val)

    return f1

   

In [128]:
study = optuna.create_study(direction='maximize')

[I 2024-10-31 14:28:48,918] A new study created in memory with name: no-name-07378d03-8c80-42a0-88df-a4d254ae7529


In [129]:
len(X_resampled)

1385501

In [130]:
len(y_resampled)

1385501

In [131]:
study.optimize(objective, n_trials=50, n_jobs=-1)

[I 2024-10-31 14:30:19,079] Trial 30 finished with value: 0.5739828664548492 and parameters: {'max_depth': 2, 'learning_rate': 0.037398347844587905, 'reg_lambda': 3.1873298658951352}. Best is trial 30 with value: 0.5739828664548492.
[I 2024-10-31 14:30:53,237] Trial 29 finished with value: 0.936289290764753 and parameters: {'max_depth': 4, 'learning_rate': 0.43667725278974295, 'reg_lambda': 4.6705979668505515}. Best is trial 29 with value: 0.936289290764753.
[I 2024-10-31 14:31:48,828] Trial 27 finished with value: 0.9492754731993941 and parameters: {'max_depth': 7, 'learning_rate': 0.9716750141455397, 'reg_lambda': 2.547157732108066}. Best is trial 27 with value: 0.9492754731993941.
[I 2024-10-31 14:32:41,639] Trial 28 finished with value: 0.9672136261542301 and parameters: {'max_depth': 9, 'learning_rate': 0.8768702106304671, 'reg_lambda': 2.210460212860359}. Best is trial 28 with value: 0.9672136261542301.
[I 2024-10-31 14:34:05,786] Trial 33 finished with value: 0.938484584723317 a

In [132]:
study.best_params

{'max_depth': 38,
 'learning_rate': 0.17759220467737735,
 'reg_lambda': 1.8231930773654492}

In [133]:
!pip install optuna[visualization]


[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [134]:
!pip install plotly



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [135]:
print(len(study.trials))

50


In [136]:
#Looking at the first trial to see if trails actually have values
print(study.trials[0])

FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.986372710702196], datetime_start=datetime.datetime(2024, 10, 31, 14, 28, 49, 150281), datetime_complete=datetime.datetime(2024, 10, 31, 14, 44, 22, 97967), params={'max_depth': 43, 'learning_rate': 0.5108136365920442, 'reg_lambda': 3.006867379729441}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'max_depth': IntDistribution(high=60, log=False, low=1, step=1), 'learning_rate': FloatDistribution(high=1.0, log=False, low=0.01, step=None), 'reg_lambda': FloatDistribution(high=10.0, log=False, low=0.0, step=None)}, trial_id=0, value=None)


In [137]:
vis.plot_optimization_history(study)

In [138]:
vis.plot_slice(study, params=['max_depth', 'learning_rate', 'reg_lambda'])

In [139]:
vis.plot_param_importances(study)

In [140]:
best_trail = study.best_trial

best_params = study.best_params

best_model = xgboost.XGBClassifier(**best_params)

best_model.fit(X_resampled, y_resampled)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.17759220467737735,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=38, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [141]:
# Best F1 score
print(f'Best F12 Score: {best_trail.value}')

Best F12 Score: 0.9872090601508191
